# Story 5.2 — Sprint 2 End-to-End Pipeline

Reuse the unchanged Sprint 1 `sample_artifacts` view (43 artifacts, 10 exact-name groups), the existing adjacent-within-name pairing (33 candidates), and the repository’s `SkillAnalyzer.scan()` and `compare_results()` methods. Adjacency is a comparison convention, **not proven ancestry**. This notebook integrates existing methods; it does not implement new detection or comparison algorithms.

## Environment and existing project imports

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(
    "/Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8"
)

sys.path.insert(0, str(PROJECT_ROOT / "src"))

from gitskills import SkillAnalyzer, compare_results

print("gitskills imported successfully")
print("Loaded from:", sys.modules["gitskills"].__file__)

gitskills imported successfully
Loaded from: /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8/src/gitskills/__init__.py


In [2]:
import json
import os
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib.pyplot as plt

from gitskills import SkillAnalyzer, compare_results


In [3]:
DEFAULT_DB_PATH = Path(r"C:\data\duckdb\agent_skills_release.db")
DB_PATH = Path(os.environ.get("GITSKILLS_DB", DEFAULT_DB_PATH))

# Resolve paths whether Jupyter starts from the repository root or notebooks/.
CWD = Path.cwd().resolve()

if CWD.name == "notebooks":
    NOTEBOOK_DIR = CWD
    PROJECT_ROOT = CWD.parent
elif (CWD / "notebooks").is_dir():
    PROJECT_ROOT = CWD
    NOTEBOOK_DIR = CWD / "notebooks"
else:
    PROJECT_ROOT = CWD
    NOTEBOOK_DIR = CWD

OUTPUT_DIR = NOTEBOOK_DIR / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
OUTPUT_PARQUET = OUTPUT_DIR / "sample_artifacts_scanner_results.parquet"

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Project root:       {PROJECT_ROOT}")
print(f"Parquet output:     {OUTPUT_PARQUET}")
print(f"Figures directory:  {FIGURES_DIR}")


Notebook directory: /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8/notebooks/end-to-end_pipeline
Project root:       /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8/notebooks/end-to-end_pipeline
Parquet output:     /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8/notebooks/end-to-end_pipeline/results/sample_artifacts_scanner_results.parquet
Figures directory:  /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/csc580-project8/notebooks/end-to-end_pipeline/figures


In [4]:
from pathlib import Path
import duckdb

DB_PATH = Path(
    "/Users/vaisnavii/csc580-data/duckdb/agent_skills_release.db"
)

print(f"Database path: {DB_PATH}")
print(f"Database exists: {DB_PATH.exists()}")

if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

conn = duckdb.connect(
    database=str(DB_PATH),
    read_only=True,
)

conn.sql("SHOW TABLES")

Database path: /Users/vaisnavii/csc580-data/duckdb/agent_skills_release.db
Database exists: True


nameVARCHAR
artifact_groupings
artifact_siblings
artifacts
mining_runs
repos
sample_artifacts


In [ ]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found: {DB_PATH}")

print(f"Database: {DB_PATH}")
print(f"Size: {DB_PATH.stat().st_size / 1024**3:.2f} GB")

In [ ]:
conn = duckdb.connect(database=str(DB_PATH), read_only=True)
duckdb_version = conn.sql("select version()").fetchall()[0][0]

print(f"Connected to DuckDB version {duckdb_version}")

In [ ]:
conn.sql("SHOW TABLES")

## 1. Check Sprint 2 sample counts

In [5]:
sample_counts = conn.sql("""
    select
        count(*) as total_artifacts,
        count(distinct file_sha) as distinct_file_sha
    from sample_artifacts
""").df()

sample_counts


,total_artifacts,distinct_file_sha
0,43,43


## 2. Inspect artifact identifiers and comparison order

In [6]:
artifact_order_fields = conn.sql("""
    select id, name, dedup_primary, file_sha, first_commit_at, repo_created_at
    from sample_artifacts
""").df()

artifact_order_fields


,id,name,dedup_primary,file_sha,first_commit_at,repo_created_at
0,618973,003-create-golang-project,1,348f7a56f87d01cb9c10b9aac82b04dbdd8c163d,NaN,2026-02-22T16:34:17Z
1,480004,003-create-golang-project,1,4c64854a9b7b41777cf7ff7aafaf882e1609a83f,NaN,2026-03-04T00:42:06Z
2,618973,003-create-golang-project,1,2941934ac0db60b9a3888ee9da82396ef063af07,NaN,2026-04-12T19:45:27Z
3,618973,003-create-golang-project,1,cc5c501649ef678b79dc30b98c62462981ea1b4f,NaN,2026-05-28T23:02:08Z
4,373553,1k-defi-module-integration,1,676a02971f9b8575506f6d628150f35dd3936f09,2026-04-15T21:39:25Z,2026-04-15T21:55:27Z
5,422274,1k-defi-module-integration,1,dc438bccc86934026118a6bf27980698d5789456,2026-05-01T00:13:08Z,2026-01-21T06:17:51Z
6,422274,1k-defi-module-integration,1,442effea7c7b6ccf9cef7e4d59bcfacce74d9d08,2026-06-03T07:00:18Z,2026-06-03T06:59:05Z
7,268864,1k-defi-module-integration,1,c6949d376537f3d07c306268572d98265a0fff88,NaN,2021-11-22T14:54:51Z
8,443341,A/B Test Setup,1,ed296841d6309ef9b0db917097b2f9f1c266fd87,NaN,2026-02-23T07:48:29Z
9,24200,A/B Test Setup,1,2c6007053f203da30f40b2017a75181a608c1b65,NaN,2026-06-05T15:55:29Z


## 3. Identify candidate artifact groups

In [7]:
name_summary = (
    artifact_order_fields.groupby("name", sort=False, dropna=False)
    .size()
    .reset_index(name="artifact_count")
)

print(f"Distinct names: {len(name_summary)}")
for name in name_summary["name"]:
    print(f"- {name}")

name_summary


Distinct names: 10
- 003-create-golang-project
- 1k-defi-module-integration
- A/B Test Setup
- argent-react-native-app-workflow
- capacitor-best-practices
- cometchat-core
- cursor-subagent-creator
- databricks-upgrade-migration
- ghostclaw
- legacy-migration-planner


,name,artifact_count
0,003-create-golang-project,4
1,1k-defi-module-integration,4
2,A/B Test Setup,2
3,argent-react-native-app-workflow,5
4,capacitor-best-practices,3
5,cometchat-core,5
6,cursor-subagent-creator,5
7,databricks-upgrade-migration,5
8,ghostclaw,5
9,legacy-migration-planner,5


## 4. Extract the sample artifacts

In [8]:
sample_artifacts = conn.sql("""
    select * from sample_artifacts
""").df()

SOURCE_COLUMNS = sample_artifacts.columns.tolist()

print(f"Loaded {len(sample_artifacts)} artifacts and {len(SOURCE_COLUMNS)} source columns.")

sample_artifacts


Loaded 43 artifacts and 24 source columns.


,id,artifact_id,repo_id,name,normalized_description,artifact_group_size,sibling_file_count,sibling_content_sha,file_sha,path,...,history_fetched,first_commit_at,composition_fetched,dedup_primary,sibling_count,has_scripts,content_sha_ok,repo_name,repo_owner,repo_created_at
0,618973,2244377,162936,003-create-golang-project,scaffolds the initial boilerplate structure fo...,3,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,348f7a56f87d01cb9c10b9aac82b04dbdd8c163d,.xdrs/agentme/edrs/application/skills/003-crea...,...,0,NaN,1,1,0,0,1,flaviostutz/filedist,flaviostutz,2026-02-22T16:34:17Z
1,480004,2386173,165213,003-create-golang-project,scaffolds the initial boilerplate structure fo...,1,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,4c64854a9b7b41777cf7ff7aafaf882e1609a83f,.xdrs/agentme/edrs/application/skills/003-crea...,...,0,NaN,1,1,0,0,1,flaviostutz/agentme,flaviostutz,2026-03-04T00:42:06Z
2,618973,1970318,162284,003-create-golang-project,scaffolds the initial boilerplate structure fo...,3,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,2941934ac0db60b9a3888ee9da82396ef063af07,.xdrs/agentme/edrs/application/skills/003-crea...,...,0,NaN,1,1,0,0,1,flaviostutz/testprompt,flaviostutz,2026-04-12T19:45:27Z
3,618973,2386167,204279,003-create-golang-project,scaffolds the initial boilerplate structure fo...,3,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,cc5c501649ef678b79dc30b98c62462981ea1b4f,.xdrs/agentme/edrs/application/skills/003-crea...,...,0,NaN,1,1,0,0,1,flaviostutz/deepworkflow,flaviostutz,2026-05-28T23:02:08Z
4,373553,2696303,63828,1k-defi-module-integration,guide for integrating new defi modules or prot...,1,6,05c4cf576dc7bbc733b2aeb31c70f0d628b70c346ae2fd...,676a02971f9b8575506f6d628150f35dd3936f09,.claude/skills/1k-defi-module-integration/SKIL...,...,1,2026-04-15T21:39:25Z,1,1,6,0,1,wanghaisheng/web3-scaffold-app-monorepo,wanghaisheng,2026-04-15T21:55:27Z
5,422274,3076631,261,1k-defi-module-integration,interactive guide for integrating new defi mod...,2,<NA>,NaN,dc438bccc86934026118a6bf27980698d5789456,Dataset/Skills/malware/1k-defi-module-integrat...,...,1,2026-05-01T00:13:08Z,1,1,1,1,1,lxyeternal/MalSkillBench,lxyeternal,2026-01-21T06:17:51Z
6,422274,2723114,3936,1k-defi-module-integration,interactive guide for integrating new defi mod...,2,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,442effea7c7b6ccf9cef7e4d59bcfacce74d9d08,.agents/skills/1k-defi-module-integration/SKIL...,...,1,2026-06-03T07:00:18Z,1,1,0,0,1,MikeCheng1208/BattleTree,MikeCheng1208,2026-06-03T06:59:05Z
7,268864,2498630,78873,1k-defi-module-integration,"onekey app defi integration for earn, borrow, ...",1,18,b6f86e95318c8321d4cd970045ef78f469cc55ff6821df...,c6949d376537f3d07c306268572d98265a0fff88,.skillshare/skills/1k-defi-module-integration/...,...,0,NaN,1,1,18,0,1,OneKeyHQ/app-monorepo,OneKeyHQ,2021-11-22T14:54:51Z
8,443341,2690162,47889,A/B Test Setup,"use this skill when setting up, improving, or ...",1,1,90d76d7e4b1dce5f90e9e3f5f4080dda182aa9ceb3c5b4...,ed296841d6309ef9b0db917097b2f9f1c266fd87,skills/ab-test-setup/SKILL.md,...,0,NaN,1,1,1,0,1,ChatAndBuild/chatchat-skills,ChatAndBuild,2026-02-23T07:48:29Z
9,24200,1244165,58628,A/B Test Setup,"use when planning, designing, or implementing ...",1,0,e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b93...,2c6007053f203da30f40b2017a75181a608c1b65,mindpalace/skills/global/marketing/ab-test-set...,...,0,NaN,1,1,0,0,1,aashutosh396/mindpalace,aashutosh396,2026-06-05T15:55:29Z


## 5. Initialize the existing security analyzer

In [9]:
analyzer = SkillAnalyzer()

rule_catalog = pd.DataFrame(
    [
        {
            "rule_id": rule.rule_id,
            "category": rule.category.value,
            "description": rule.description,
        }
        for rule in analyzer.rules
    ]
)

RULE_CATEGORY = dict(zip(rule_catalog["rule_id"], rule_catalog["category"]))
RULE_DESCRIPTION = dict(zip(rule_catalog["rule_id"], rule_catalog["description"]))

rule_catalog

,rule_id,category,description
0,CMD-001,command_execution,Shell command block detected
1,CMD-002,command_execution,Command invocation detected
2,NET-001,network_access,HTTP or HTTPS URL detected
3,NET-002,network_access,Network client usage detected
4,FS-001,filesystem_access,File-system command detected
5,FS-002,filesystem_access,File-writing API detected
6,FS-003,filesystem_access,File-reading command or API detected
7,CRED-001,credential_access,Credential or secret reference detected
8,CRED-002,credential_access,Credential file reference detected
9,EXT-001,external_code_execution,Downloaded content is piped to a shell interpr...


## 5. Validate sample and create traceable comparison candidates

The database view defines the input population. We preserve its returned row order and group by exact artifact name without sorting. The membership hash records that order for reproducibility; DuckDB views do not inherently guarantee row order across executions.

In [15]:
import hashlib, dataclasses
from enum import Enum
from collections import Counter
from datetime import datetime, timezone

RESULTS_DIR = PROJECT_ROOT / "results" / "sprint2"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def json_safe(value):
    if isinstance(value, Enum): return value.value
    if dataclasses.is_dataclass(value):
        return {f.name: json_safe(getattr(value, f.name)) for f in dataclasses.fields(value)}
    if isinstance(value, dict): return {str(json_safe(k)): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_safe(v) for v in value]
    if isinstance(value, (set, frozenset)): return sorted(json_safe(v) for v in value)
    if value is None or isinstance(value, (str, int, float, bool)): return value
    if hasattr(value, "item"): return json_safe(value.item())
    raise TypeError(f"Unsupported output value: {type(value).__name__}")

def save_json(name, value):
    path = RESULTS_DIR / name
    path.write_text(json.dumps(json_safe(value), indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    json.loads(path.read_text(encoding="utf-8"))
    return path

def source(row):
    def value(*names):
        for name in names:
            if name in row.index:
                v = row[name]
                if v is not None and not (isinstance(v, str) and not v.strip()):
                    try:
                        if bool(pd.isna(v)): continue
                    except (TypeError, ValueError): pass
                    return str(v)
        return None
    artifact_id, sha = value("id", "artifact_id"), value("file_sha")
    return dict(artifact_key=f"{artifact_id}:{sha}" if artifact_id and sha else None,
                artifact_id=artifact_id, file_sha=sha, name=value("name"),
                repository_id=value("repo_id", "repository_id"),
                repository_name=value("repo_name", "repository_name", "full_name"),
                path=value("path", "file_path", "artifact_path"))

sample_refs = [source(row) for _, row in sample_artifacts.iterrows()]
assert len(sample_refs) == 43
assert sample_artifacts["name"].nunique(dropna=False) == 10
assert len({r["artifact_key"] for r in sample_refs}) == 43
assert all(r["artifact_key"] for r in sample_refs)
# Keep group and within-group order exactly as returned by the sample view.
candidate_pairs = [(name, a, b) for name, group in sample_artifacts.groupby("name", sort=False, dropna=False)
                   for a, b in zip(group.index.tolist(), group.index.tolist()[1:])]
assert len(candidate_pairs) == 33
membership_hash = hashlib.sha256(json.dumps(sample_refs, sort_keys=True).encode()).hexdigest()
save_json("sample_membership.json", sample_refs)
save_json("sample_manifest.json", dict(view="sample_artifacts", sample_origin="Sprint 1 reused for Sprint 2",
    artifacts=43, name_groups=10, distinct_file_shas=int(sample_artifacts.file_sha.nunique()),
    ordered_membership_sha256=membership_hash, database_filename=DB_PATH.name,
    pairing="adjacent within exact name in view-returned order; not proven ancestry"))
print("Sample:", len(sample_refs), "artifacts;", len(candidate_pairs), "candidate pairs")

Sample: 43 artifacts; 33 candidate pairs


## 6. Scan each eligible artifact using the existing security analyzer

Each sample artifact is considered independently. Invalid records are logged rather than silently removed. The scanner itself is unchanged.

In [16]:
scan_results, profiles, artifact_log = {}, [], []
for i, (_, row) in enumerate(sample_artifacts.iterrows()):
    ref = sample_refs[i]
    content = row.get("content")
    reason = ("missing_file_sha" if not ref["file_sha"] else
              "missing_source_identifier" if not ref["artifact_id"] else
              "missing_content" if content is None or (isinstance(content, float) and pd.isna(content)) else
              "non_text_content" if not isinstance(content, str) else
              "empty_content" if not content.strip() else None)
    status, error = "skipped" if reason else "scanned", None
    if not reason:
        try:
            result = analyzer.scan(content)  # Existing repository implementation
            scan_results[ref["artifact_key"]] = result
            profiles.append(dict(source=ref, filename=result.filename, source_id=result.source_id,
                capabilities=sorted(c.value for c in result.profile.capabilities),
                rule_counts=result.rule_counts(), rule_matches=json_safe(result.rule_matches),
                evidence=json_safe(result.evidence)))
        except Exception as exc:
            status, reason, error = "error", "scanner_error", f"{type(exc).__name__}: {exc}"
    artifact_log.append(dict(**ref, stage="scan", status=status, reason=reason, error=error))
save_json("security_profiles.json", profiles)
pd.DataFrame(artifact_log).to_csv(RESULTS_DIR / "artifact_processing_log.csv", index=False)
print("Successfully scanned:", len(profiles), "of", len(sample_refs))

Successfully scanned: 43 of 43


## 7. Compare existing adjacent pairs using the repository comparator

`compare_results()` supplies introduced categories, rule deltas, and evidence additions/removals. The notebook only associates these results with their original artifact references.

In [17]:
changes, comparison_log = [], []
for order, (name, a, b) in enumerate(candidate_pairs, 1):
    base, derived = sample_refs[a], sample_refs[b]
    left, right = scan_results.get(base["artifact_key"]), scan_results.get(derived["artifact_key"])
    status, reason, error, detail, flagged = "compared", None, None, None, False
    if left is None or right is None:
        status, reason = "skipped", "missing_profile_for_comparison"
    else:
        try:
            comparison = compare_results(left, right)  # Existing repository implementation
            detail = dict(introduced=comparison.introduced.to_dict(),
                rule_deltas={d.rule_id: d.to_dict() for d in comparison.rule_deltas},
                rule_match_delta=comparison.rule_match_delta,
                evidence_added=json_safe(comparison.evidence_added),
                evidence_removed=json_safe(comparison.evidence_removed))
            flagged = any(d.delta > 0 for d in comparison.rule_deltas)
        except Exception as exc:
            status, reason, error = "error", "comparison_error", f"{type(exc).__name__}: {exc}"
    changes.append(dict(pair_order=order, name=name, base=base, derived=derived,
        status=status, reason=reason, error=error, flagged=flagged, security_change=detail))
    comparison_log.append(dict(pair_order=order, name=name,
        base_artifact_key=base["artifact_key"], derived_artifact_key=derived["artifact_key"],
        base_repository=base["repository_name"], derived_repository=derived["repository_name"],
        base_path=base["path"], derived_path=derived["path"],
        base_file_sha=base["file_sha"], derived_file_sha=derived["file_sha"],
        status=status, reason=reason, error=error))
save_json("security_changes.json", changes)
pd.DataFrame(comparison_log).to_csv(RESULTS_DIR / "comparison_processing_log.csv", index=False)
print("Compared:", sum(c["status"] == "compared" for c in changes), "of", len(changes))

Compared: 33 of 33


## 8. Validate traceability, accounting, and exported results

A successful status means the integrated run and validations completed, not that every artifact or pair was eligible. Skips and errors remain visible in the logs.

In [18]:
a_counts = Counter(x["status"] for x in artifact_log)
p_counts = Counter(x["status"] for x in changes)
keys = {r["artifact_key"] for r in sample_refs}
assert sum(a_counts[s] for s in ("scanned", "skipped", "error")) == 43
assert sum(p_counts[s] for s in ("compared", "skipped", "error")) == 33
assert a_counts["scanned"] > 0, "No successful security scans"
assert len(profiles) == a_counts["scanned"]
assert all(p["source"]["artifact_key"] in keys for p in profiles)
assert all(c["base"]["artifact_key"] in keys and c["derived"]["artifact_key"] in keys for c in changes)
assert all(c["base"]["artifact_key"] in scan_results and c["derived"]["artifact_key"] in scan_results for c in changes if c["status"] == "compared")
assert all(x["reason"] for x in artifact_log if x["status"] != "scanned")
assert all(x["reason"] for x in changes if x["status"] != "compared")
assert all(not (r["path"] or "").startswith("/") for r in sample_refs), "Absolute source path found"
files = ["sample_manifest.json", "sample_membership.json", "security_profiles.json",
         "security_changes.json", "artifact_processing_log.csv", "comparison_processing_log.csv"]
for name in files:
    assert (RESULTS_DIR / name).is_file(), name
    if name.endswith(".json"): json.loads((RESULTS_DIR / name).read_text(encoding="utf-8"))
summary = dict(pipeline_status="success", sample_artifacts=43, sample_name_groups=10,
    membership_sha256=membership_hash, security_profiles_scanned=a_counts["scanned"],
    security_profiles_skipped=a_counts["skipped"], security_profiles_errors=a_counts["error"],
    candidate_comparisons=33, comparisons_scanned=p_counts["compared"],
    comparisons_skipped=p_counts["skipped"], comparisons_errors=p_counts["error"],
    flagged_comparisons=sum(bool(c["flagged"]) for c in changes),
    generated_at_utc=datetime.now(timezone.utc).isoformat())
save_json("execution_summary.json", summary)
print(json.dumps(summary, indent=2))

{
  "pipeline_status": "success",
  "sample_artifacts": 43,
  "sample_name_groups": 10,
  "membership_sha256": "edacbd69b656321189677e807c33cdbe88cbe4b63ec0e9c615370ddd6c7b11fa",
  "security_profiles_scanned": 43,
  "security_profiles_skipped": 0,
  "security_profiles_errors": 0,
  "candidate_comparisons": 33,
  "comparisons_scanned": 33,
  "comparisons_skipped": 0,
  "comparisons_errors": 0,
  "flagged_comparisons": 14,
  "generated_at_utc": "2026-10-23T17:26:38.495909+00:00"
}


## 9. Reproduce the notebook

1. From a clean checkout, install the project dependencies using the repository’s existing environment instructions (including `duckdb`, `pandas`, `matplotlib`, and the `gitskills` package).
2. Obtain the prepared **Sprint 1 DuckDB database**; it must already contain the unchanged `sample_artifacts` view. The notebook does not create or modify that view.
3. Set `GITSKILLS_DB` to the database path before starting Jupyter.
4. Open this notebook in the repository and select the project Python environment. Run all cells in order.
5. Confirm the sample reports **43 artifacts**, **10 name groups**, and **33 candidate comparisons**.
6. Check `notebooks/end-to-end_pipeline/results/sprint2/execution_summary.json` for `pipeline_status: success`; inspect the CSV logs for skipped/error reasons and JSON outputs for source identifiers and evidence.

**Limitations:** The notebook uses view-returned order; the membership hash records the actual order observed. It does not prove ancestry. A successful clean-environment run must be performed on a machine with the prepared database; no such run is claimed here.

In [14]:
conn.close()
print("DuckDB connection closed.")

DuckDB connection closed.
